In [ ]:
using SparseArrays, LinearAlgebra, KrylovKit, Random, CairoMakie, JLD2

In [ ]:
function apply_sigma_z!(ψ_out::Vector{ComplexF32}, ψ_in::Vector{ComplexF32}, site::Int)
    @inbounds for b in 0:(length(ψ_in)-1)
        sign = ((b >> (site-1)) & 1) == 1 ? -1.0f0 : 1.0f0
        ψ_out[b+1] = sign * ψ_in[b+1]
    end
    return ψ_out
end
apply_sigma_z(ψ::Vector{ComplexF32}, site::Int) = apply_sigma_z!(similar(ψ), ψ, site)

function apply_sigma_x!(ψ_out::Vector{ComplexF32}, ψ_in::Vector{ComplexF32}, site::Int)
    @inbounds for b in 0:(length(ψ_in)-1)
        b_flipped = b ⊻ (1 << (site - 1))
        
        ψ_out[b+1] = ψ_in[b_flipped+1]
    end
    return ψ_out
end
apply_sigma_x(ψ::Vector{ComplexF32}, site::Int) = apply_sigma_x!(similar(ψ), ψ, site)

function build_hamiltonian(N::Int, J::Float32, λ::Float32, h::Float32, hA::Float32, g::Float32, gA::Float32)
    # Total spins is N (chain) + 1 (ancilla)
    N_total = N + 1
    dim = 1 << N_total
    
    num_entries = dim * (N_total + 1) 
    I = Vector{Int}(undef, num_entries)
    J_idx = Vector{Int}(undef, num_entries)
    V = Vector{ComplexF32}(undef, num_entries)
    
    idx = 1
    for b in 0:(dim-1)
        diag_val = 0.0f0
        
        # Extract Ancilla Z projection (spin N+1, which is bit N)
        sz_ancilla = ((b >> N) & 1) == 1 ? -0.5f0 : 0.5f0
        
        # ZZ interactions on the chain (open boundary conditions: 1 to N)
        for j in 1:(N-1)
            sz_j   = ((b >> (j-1)) & 1) == 1 ? -0.5f0 : 0.5f0
            sz_jp1 = ((b >> j) & 1) == 1 ? -0.5f0 : 0.5f0
            diag_val += J * sz_j * sz_jp1
        end
        
        # λ coupling from Ancilla to ALL N spins in the chain
        for j in 1:N
            sz_j = ((b >> (j-1)) & 1) == 1 ? -0.5f0 : 0.5f0
            diag_val += λ * sz_j * sz_ancilla
        end
        
        # Z fields for the chain (1 to N)
        for j in 1:N
            sz_j = ((b >> (j-1)) & 1) == 1 ? -0.5f0 : 0.5f0
            diag_val -= g * sz_j
        end
        
        # Z field for the ancilla
        diag_val -= gA * sz_ancilla
        
        # Store diagonal element
        I[idx] = b + 1; J_idx[idx] = b + 1; V[idx] = diag_val
        idx += 1
        
        # X fields (off-diagonal bit flips) for the chain (1 to N)
        for j in 1:N
            b_flipped = b ⊻ (1 << (j-1))
            I[idx] = b_flipped + 1; J_idx[idx] = b + 1; V[idx] = -h * 0.5f0
            idx += 1
        end
        
        # X field for the ancilla (bit N)
        b_flipped = b ⊻ (1 << N)
        I[idx] = b_flipped + 1; J_idx[idx] = b + 1; V[idx] = -hA * 0.5f0
        idx += 1
    end
    
    return sparse(I, J_idx, V, dim, dim)
end

function build_hamiltonian_transverse(N::Int, J::Float32, λ::Float32, h::Float32, hA::Float32, g::Float32, gA::Float32)
    # Total spins is N (chain) + 1 (ancilla)
    N_total = N + 1
    dim = 1 << N_total
    
    # 1 (diagonal) + N (chain X fields) + 1 (ancilla X field) + N (XX couplings)
    # Total = 2N + 2 entries per row
    num_entries = dim * (2 * N + 2) 
    I = Vector{Int}(undef, num_entries)
    J_idx = Vector{Int}(undef, num_entries)
    V = Vector{ComplexF32}(undef, num_entries)
    
    idx = 1
    for b in 0:(dim-1)
        diag_val = 0.0f0
        
        # Extract Ancilla Z projection (spin N+1, which is bit N)
        sz_ancilla = ((b >> N) & 1) == 1 ? -0.5f0 : 0.5f0
        
        # ZZ interactions on the chain (open boundary conditions: 1 to N)
        for j in 1:(N-1)
            sz_j   = ((b >> (j-1)) & 1) == 1 ? -0.5f0 : 0.5f0
            sz_jp1 = ((b >> j) & 1) == 1 ? -0.5f0 : 0.5f0
            diag_val += J * sz_j * sz_jp1
        end
        
        # Z fields for the chain (1 to N)
        for j in 1:N
            sz_j = ((b >> (j-1)) & 1) == 1 ? -0.5f0 : 0.5f0
            diag_val -= g * sz_j
        end
        
        # Z field for the ancilla
        diag_val -= gA * sz_ancilla
        
        # Store diagonal element
        I[idx] = b + 1; J_idx[idx] = b + 1; V[idx] = diag_val
        idx += 1
        
        # X fields (off-diagonal bit flips) for the chain (1 to N)
        for j in 1:N
            b_flipped = b ⊻ (1 << (j-1))
            I[idx] = b_flipped + 1; J_idx[idx] = b + 1; V[idx] = -h * 0.5f0
            idx += 1
        end
        
        # X field for the ancilla (bit N)
        b_flipped = b ⊻ (1 << N)
        I[idx] = b_flipped + 1; J_idx[idx] = b + 1; V[idx] = -hA * 0.5f0
        idx += 1
        
        # λ coupling from Ancilla to ALL N spins in the chain (X-X coupling)
        for j in 1:N
            b_flipped_xx = b ⊻ (1 << (j-1)) ⊻ (1 << N)
            I[idx] = b_flipped_xx + 1; J_idx[idx] = b + 1; V[idx] = λ * 0.25f0
            idx += 1
        end
    end
    
    return sparse(I, J_idx, V, dim, dim)
end

# OTOC Computation

In [ ]:
function simulate_otoc(N::Int, J::Real, λ::Real, tmax::Real, dt::Real; h::Real=1.0, hA::Real=1.0, g::Real=0.0, gA::Real=0.0)
    original_blas_threads = LinearAlgebra.BLAS.get_num_threads()
    LinearAlgebra.BLAS.set_num_threads(1)
    
    H = build_hamiltonian_transverse(N, Float32(J), Float32(λ), Float32(h), Float32(hA), Float32(g), Float32(gA))
    mode = 'x'
    
    dim = 1 << (N+1)
    ψ_inf = randn(ComplexF32, dim)
    ψ_inf ./= norm(ψ_inf) 
    
    times = Float32(dt):Float32(dt):Float32(tmax)
    otoc_matrix = zeros(ComplexF32, N + 1, length(times))
    
    Threads.@threads for j in 1:N+1
        ψ_0_t = copy(ψ_inf)
        ψ_V_t = mode=='x' ? apply_sigma_x(ψ_inf, j) : apply_sigma_z(ψ_inf, j) 
        
        ψ_W_0_t = similar(ψ_0_t)
        ψ_W_V_t = similar(ψ_V_t)
        
        for (i, t) in enumerate(times)
            
            ψ_0_t, _ = exponentiate(H, -1im * Float32(dt), ψ_0_t; ishermitian=true)
            ψ_V_t, _ = exponentiate(H, -1im * Float32(dt), ψ_V_t; ishermitian=true)

            # Apply W = sigma_1^z 
            if mode == 'x'
                apply_sigma_x!(ψ_W_0_t, ψ_0_t, 1)
                apply_sigma_x!(ψ_W_V_t, ψ_V_t, 1)
            else
                apply_sigma_z!(ψ_W_0_t, ψ_0_t, 1)
                apply_sigma_z!(ψ_W_V_t, ψ_V_t, 1)
            end

            
            # Backward evolution to time 0 
            back_0, _ = exponentiate(H, 1im * t, ψ_W_0_t; ishermitian=true)
            back_V, _ = exponentiate(H, 1im * t, ψ_W_V_t; ishermitian=true)
            
            # Measure: 1 - Re(<back_0 | V | back_V>)
            if mode == 'x'
                apply_sigma_x!(ψ_W_V_t, back_V, j)
            else
                apply_sigma_z!(ψ_W_V_t, back_V, j)
            end
            
            otoc_matrix[j,i] = (back_0' * ψ_W_V_t)
        end
    end
    
    LinearAlgebra.BLAS.set_num_threads(original_blas_threads)
    
    return otoc_matrix
end

In [ ]:
N = 8;
J = 1.;
λ = 0.;

tmax = 40; dt = 0.5; tarr = collect(dt:dt:tmax)

data = simulate_otoc(N, J, λ, tmax, dt; h=1.05, hA=1.05, g=0.45, gA=0.45);

In [ ]:
fig = Figure(size = (800, 600))
ax = Axis(fig[1, 1], title = "",
    xticks = (cat(collect(0:2:N-1),N,dims=1),cat(["$j" for j in 0:2:N-1],"A",dims=1)), xticklabelsize = 18, xlabel = L"r", xlabelsize = 24,
    yticklabelsize = 18, ylabel = L"Jt", ylabelsize = 24)
hm = heatmap!(ax, collect(0:N), collect(dt:dt:tmax), (1 .- real(data))/2, colormap = :plasma, colorrange = (0,1.01))
vlines!(ax, [N-1/2], color = :white, linewidth = 2, label = "");
text!(ax, 0.85, 0.04, 
      text = L"\lambda = %$(λ)", 
      space = :relative, 
      align = (:right, :bottom), 
      fontsize = 28, 
      color = :white)

Colorbar(fig[1, 2], hm, label = "", ticks = collect(0:0.2:1), ticklabelsize = 18)

display(fig);

## OTOC traces vs. $r$

In [ ]:
fig = Figure(size = (800, 600))
ax = Axis(fig[1, 1], title = "", limits = (nothing, 10, 1e-5, nothing),
    xlabel = L"Jt", xlabelsize = 24, xscale = log10, 
    xticklabelsize = 18, xminorticks = IntervalsBetween(10), xminorticksvisible = true, 
    xgridvisible = true, xminorgridvisible = true,
    ylabel = L"C(r,t)", ylabelsize = 24, yscale = log10, yticks = ([10. ^(-j) for j in 0:5],["1e-$(j)" for j in 0:5]), 
    yticklabelsize = 18, yminorticks = IntervalsBetween(10), yminorticksvisible = true, 
    ygridvisible = true, yminorgridvisible = true)

rarr = 0:N-1
carr = [(Makie.resample_cmap(:plasma, length(rarr)))...]

## clean data 
data[real(data) .> 1] .= 1.0 + 1.0im;

fit_a = Float64[]
fit_b = Float64[]

for (j, r) in enumerate(rarr)
    y_data = (1 .- real(data[1+r, :])) ./ 2
    
    lines!(ax, tarr, y_data, color = carr[j], linewidth = 2, label = L"r = %$(r)")
    
    # --- Power Law Fitting ---
    # Filter out late-time saturation (e.g., y > 0.1) so the power law only captures the growth front.
    valid_idx = (1e-5 .< y_data .< 0.05)
    
    t_valid = tarr[valid_idx]
    y_valid = y_data[valid_idx]

    if length(t_valid) > 1
        # Formulate as linear regression: log10(y) = log10(a) + b * log10(t)
        X = hcat(ones(length(t_valid)), log10.(t_valid))
        Y = log10.(y_valid)
        
        coeffs = X \ Y 
        
        a = 10^(coeffs[1])
        b = coeffs[2]
        
        push!(fit_a, a)
        push!(fit_b, b)
        
        # lines!(ax, t_valid, a .* (t_valid .^ b), color = carr[j], linestyle = :dash, linewidth = 4)
    else
        push!(fit_a, NaN)
        push!(fit_b, NaN)
    end
end
lines!(ax, tarr, (1 .- real(data[end, :])) ./ 2, linestyle = :dash, color = :cyan, linewidth = 2, label = "Ancilla")

text!(ax, 0.05, 0.96, 
      text = L"\lambda = %$(λ)", 
      space = :relative, 
      align = (:left, :top), 
      fontsize = 28)
Legend(fig[1, 2], ax, legendsize = 32, "", framevisible = false, labelsize = 24)

display(fig);


## OTOC Traces vs. $\lambda$

In [ ]:
function simulate_otoc_trace(j::Int, k::Int, N::Int, tmax::Real, dt::Real, λarr::Vector{Float64}; J::Real = 1.0, h::Real=1.0, hA::Real=1.0, g::Real=0.0, gA::Real=0.0)
    @assert j ≤ N+1 "j must be ≤ (N + 1)"
    @assert k ≤ N+1 "k must be ≤ (N + 1)"

    times = Float32(dt):Float32(dt):Float32(tmax)
    dim = 1 << (N+1)
    
    original_blas_threads = LinearAlgebra.BLAS.get_num_threads()
    LinearAlgebra.BLAS.set_num_threads(1)
    
    otoc_matrix = zeros(ComplexF32, length(λarr), length(times))
    
    ψ_inf_global = randn(ComplexF32, dim)
    ψ_inf_global ./= norm(ψ_inf_global) 
    
    Threads.@threads for l in 1:length(λarr)
        λ = λarr[l]
        H = build_hamiltonian(N, Float32(J), Float32(λ), Float32(h), Float32(hA), Float32(g), Float32(gA))    
        
        ψ_0_t = copy(ψ_inf_global)
        ψ_V_t = apply_sigma_z(ψ_inf_global, k) 
        
        ψ_W_0_t = similar(ψ_0_t)
        ψ_W_V_t = similar(ψ_V_t)
        
        for (i, t) in enumerate(times)
            ψ_0_t, _ = exponentiate(H, -1im * Float32(dt), ψ_0_t; ishermitian=true, krylovdim=16)
            ψ_V_t, _ = exponentiate(H, -1im * Float32(dt), ψ_V_t; ishermitian=true, krylovdim=16)

            # Apply W = sigma_j^z (in-place)
            apply_sigma_z!(ψ_W_0_t, ψ_0_t, j)
            apply_sigma_z!(ψ_W_V_t, ψ_V_t, j)

            # Backward evolution to time 0 
            back_0, _ = exponentiate(H, 1im * t, ψ_W_0_t; ishermitian=true, krylovdim=60)
            back_V, _ = exponentiate(H, 1im * t, ψ_W_V_t; ishermitian=true, krylovdim=60)
            
            # Measure: < back_0 | V | back_V >
            apply_sigma_z!(ψ_W_V_t, back_V, k)
            
            otoc_matrix[l, i] = dot(back_0, ψ_W_V_t)
        end
    end
    
    LinearAlgebra.BLAS.set_num_threads(original_blas_threads)
    
    return otoc_matrix
end

In [ ]:
N = 6;
j = 1;
k = N+1;

λarr = collect(10 .^ range(log10(0.5),1.5,length=20));

tmax = 40.; dt = 0.1; tarr = collect(dt:dt:tmax)

data = simulate_otoc_trace(j, k, N, tmax, dt, λarr; J = 1., h=1.05, hA=1.05, g=0.45, gA=0.45);

In [ ]:
fig = Figure(size = (800, 600))
ax = Axis(fig[1, 1], title = "", limits = (nothing, nothing, 1e-4, nothing),
    xlabel = L"\lambda t", xlabelsize = 24, xscale = log10, xticks = ([10. ^(j) for j in -1:2],["1e$(j)" for j in -1:2]),
    xticklabelsize = 18, xminorticks = IntervalsBetween(10), xminorticksvisible = true, 
    xgridvisible = true, xminorgridvisible = true,
    ylabel = L"C_{zz}(r,t)", ylabelsize = 24, yscale = log10, yticks = ([10. ^(-j) for j in 0:3],["1e-$(j)" for j in 0:3]), 
    yticklabelsize = 18, yminorticks = IntervalsBetween(10), yminorticksvisible = true, 
    ygridvisible = true, yminorgridvisible = true)

λ_ids = 1:length(λarr);
carr = [(Makie.resample_cmap(:plasma, length(λ_ids)))...]

## clean data
data[real(data) .> 1] .= 1.0 + 1.0im;

fit_a = Float64[]
fit_b = Float64[]

for (j, l) in enumerate(λ_ids)
    y_data = (1 .- real(data[l, :])) ./ 2
    
    lines!(ax, λarr[l] .* tarr, y_data, color = carr[j], linewidth = 2, label = L"\lambda = %$(round(λarr[l],digits=2))")
    
    # Filter out late-time saturation (e.g., y > 0.1) so the power law only captures the growth front.
    valid_idx = ( 2π .< (λarr[l] .* tarr) )
    
    t_valid = tarr[valid_idx]
    y_valid = y_data[valid_idx]

    if length(t_valid) > 1
        # Formulate as linear regression: log10(y) = log10(a) + b * log10(t)
        X = hcat(ones(length(t_valid)), log10.(t_valid))
        Y = log10.(y_valid)
        
        coeffs = X \ Y 
        
        a = 10^(coeffs[1])
        b = coeffs[2]
        
        push!(fit_a, a)
        push!(fit_b, b)
        
        lines!(ax, λarr[l] .* t_valid, a .* (t_valid .^ b), color = carr[j], linestyle = :dash, linewidth = 4)
    else
        push!(fit_a, NaN)
        push!(fit_b, NaN)
    end
end

text!(ax, 0.05, 0.96, 
    #text = L"r = %$(abs(j-k))", 
    text = "Ancilla",
    space = :relative, 
    align = (:left, :top), 
    fontsize = 28)
Legend(fig[1, 2], ax, legendsize = 32, "", framevisible = false, labelsize = 24)

display(fig);


## Finite Size Scaling 

In [ ]:
λarr = round.( (10 .^ range(-1,1.5,length=100)), digits = 2)[15:end]
N_arr = 6:14

fit_a_mat    = fill(NaN, length(λarr), length(N_arr))
fit_b_mat    = fill(NaN, length(λarr), length(N_arr))
err_loga_mat = fill(NaN, length(λarr), length(N_arr))
err_b_mat    = fill(NaN, length(λarr), length(N_arr))

carr = [(Makie.resample_cmap(:plasma, length(λarr)))...]

fig1 = Figure(size = (800, 600))
ax1 = Axis(fig1[1, 1], title = "", limits = (1e-1, 10^(2.5), 1e-5, 1e0),
    xlabel = L"\lambda t", xlabelsize = 24, xscale = log10, xticks = ([10. ^(k) for k in -1:2],["1e$(k)" for k in -1:2]),
    xticklabelsize = 18, xminorticks = IntervalsBetween(10), xminorticksvisible = true, 
    xgridvisible = true, xminorgridvisible = true,
    ylabel = L"C_{zz}(r,t)", ylabelsize = 24, yscale = log10, yticks = ([10. ^(-k) for k in 0:5],["1e-$(k)" for k in 0:5]), 
    yticklabelsize = 18, yminorticks = IntervalsBetween(10), yminorticksvisible = true, 
    ygridvisible = true, yminorgridvisible = true)

for (j, N) in enumerate(N_arr)

    for (i, l) in enumerate(λarr)
        filepath = "../Ising RA Data/OTOCS_Czz_sweep_5.20.26/OTOCS_Czz_N$(N)/IsingRA_Czz_trace_N=$(N)_r=$(N-1)_lambda=$(l)_J=1.0_h=1.05_g=0.45.jld2"
        
        if !isfile(filepath)
            continue
        end

        tarr, otoc_data = load_object(filepath)
        otoc_data[real.(otoc_data) .> 1] .= 1.0 + 1.0im
        y_data = (1 .- real(otoc_data)) ./ 2

        if j==length(N_arr) && i%3==1
            lines!(ax1, l .* tarr, real(y_data), color = carr[i], linewidth = 2, label = i == 1 ? L"\lambda = %$(l)" : nothing)
        end
        
        valid_idx = ( 2π .< (l .* tarr) )
        t_valid = tarr[valid_idx]
        y_valid = y_data[valid_idx]

        if length(t_valid) > 2
            X = hcat(ones(length(t_valid)), log10.(t_valid))
            Y = log10.(y_valid)
            
            # Solve OLS
            coeffs = X \ Y 
            c1, c2 = coeffs[1], coeffs[2]
            
            a = 10^c1
            b = c2
            
            fit_a_mat[i, j] = a
            fit_b_mat[i, j] = b
            
            # Error estimation via standard OLS covariance matrix
            residuals = Y .- X * coeffs
            dof = length(t_valid) - 2
            s_sq = sum(residuals.^2) / dof
            cov_matrix = s_sq * inv(X' * X)
            
            err_c1 = sqrt(cov_matrix[1, 1])
            err_c2 = sqrt(cov_matrix[2, 2])
            
            # Propagate error: variance in ln(a) given c1 = log_10(a)
            err_loga_mat[i, j] = err_c1 * log(10)
            err_b_mat[i, j] = err_c2
            
            if j==length(N_arr) && i%3==1
                lines!(ax1, l .* t_valid, a .* (t_valid .^ b), color = carr[i], linestyle = :dash, linewidth = 4)
            end
        end
    end
end
text!(ax1, 0.05, 0.96, text = L"r = 13", 
    #text = "Ancilla",
    space = :relative, align = (:left, :top), fontsize = 28)
Colorbar(fig1[1, 0], colormap = :plasma, limits = (1e-1, 10^(1.5)), scale = log10, ticklabelsize = 24)

display(fig1) 

In [ ]:
fig2 = Figure(size = (800, 600))
ax2_1 = Axis(fig2[1, 1], title = "", limits = (10^(-0.6), nothing, -12, 0),
    xlabel = L"\lambda", xlabelsize = 24, xscale = log10, xticks = ([10. ^(j) for j in -1:1], ["1e$(j)" for j in -1:1]),
    xticklabelsize = 18, xminorticks = IntervalsBetween(10), xminorticksvisible = true, 
    xgridvisible = true, xminorgridvisible = true,
    ylabel = L"\log[\alpha]", ylabelsize = 24, ylabelcolor = :blue, 
    yticklabelsize = 18, yticklabelcolor = :blue, yminorticks = IntervalsBetween(5), yminorticksvisible = true, 
    ygridvisible = false, yminorgridvisible = false)

ax2_2 = Axis(fig2[1, 1], yticklabelcolor = :red, yaxisposition = :right,limits = (10^(-0.6), nothing, -0.1, 2.0),
    ylabel = L"\beta", ylabelsize = 24, ylabelcolor = :red, ygridvisible = false, yminorgridvisible = false,
    xscale = log10
)
hidespines!(ax2_2)
hidexdecorations!(ax2_2)
linkxaxes!(ax2_1, ax2_2)

n_colors = length(N_arr)
blues_cmap = cgrad(:Blues, n_colors + 3)[4:end]
reds_cmap  = cgrad(:Reds,  n_colors + 3)[4:end]

for (j, N) in enumerate(N_arr)
    # Extract data for the current N
    y_a = log.(fit_a_mat[:, j])
    err_a = err_loga_mat[:, j]
    
    y_b = fit_b_mat[:, j]
    err_b = err_b_mat[:, j]
    
    # Filter out NaNs 
    v_idx = .!isnan.(y_a)
    λ_val = λarr[v_idx]
    
    if isempty(λ_val)
        continue
    end

    errorbars!(ax2_1, λ_val, y_a[v_idx], err_a[v_idx], color = blues_cmap[j], whiskerwidth = 4)
    errorbars!(ax2_2, λ_val, y_b[v_idx], err_b[v_idx], color = reds_cmap[j], whiskerwidth = 4)

    lines!(ax2_1, λ_val, y_a[v_idx], color = blues_cmap[j], linewidth = 3)
    lines!(ax2_2, λ_val, y_b[v_idx], color = reds_cmap[j], linewidth = 2)
end

max_N_val = λarr[λarr .> sqrt(maximum(N_arr))]
lines!(ax2_1, max_N_val, 4*log.(max_N_val .^(-1)) .+ 2, color = :cyan, linewidth = 5, linestyle = :dash, label = L"\sim 1/\lambda")
text!(ax2_1, 0.8, 0.4, text = L"\sim \lambda^{-1}", fontsize = 28, space = :relative, align = (:left, :top))

Colorbar(fig2[1, 0], colormap = blues_cmap, limits = (6,14), ticklabelsize = 24)
Colorbar(fig2[1, 2], colormap = reds_cmap, limits = (6,14), ticklabelsize = 24)

display(fig2)

# Open Boundary Ring-Ancilla Entropy 

In [ ]:
BLAS.set_num_threads(Int(Sys.CPU_THREADS))

function initialize_plus_y_state(N_total::Int)
    dim = 1 << N_total
    ψ = Vector{ComplexF32}(undef, dim)
    norm_factor = 1.0f0 / sqrt(Float32(dim))
    
    @inbounds for b in 0:(dim-1)
        k = count_ones(b)
        rem = k % 4
        if rem == 0
            ψ[b+1] = ComplexF32(norm_factor, 0.0f0)
        elseif rem == 1
            ψ[b+1] = ComplexF32(0.0f0, norm_factor)
        elseif rem == 2
            ψ[b+1] = ComplexF32(-norm_factor, 0.0f0)
        else
            ψ[b+1] = ComplexF32(0.0f0, -norm_factor)
        end
    end
    return ψ
end

@inline function compute_entropy_from_eigvals(λs::Vector{Float32})
    entropy = 0.0f0
    @inbounds for i in eachindex(λs)
        λ_i = λs[i]
        if λ_i > 1e-10
            entropy -= λ_i * log2(λ_i)
        end
    end
    return entropy
end

function compute_mutual_information(ψ::Vector{ComplexF32}, N::Int)
    N_total = N + 1
    
    N_half = N ÷ 2
    dim_L2 = 1 << N_half
    dim_rest = 1 << (N_total - N_half)
    
    A_L2 = reshape(ψ, dim_L2, dim_rest)
    rho_L2 = Hermitian(A_L2 * A_L2') 
    λs_L2 = eigvals(rho_L2)
    S_L2 = compute_entropy_from_eigvals(λs_L2)
    
    dim_chain = 1 << N
    A_A = reshape(ψ, dim_chain, 2)
    
    rho_A = Hermitian(A_A' * A_A)
    λs_A = eigvals(rho_A)
    S_A = compute_entropy_from_eigvals(λs_A)
    
    return 2.0f0 * S_L2 - S_A
end

function simulate_mutual_information(N::Int, J::Float32, λ_vals::Vector{Float32}, 
                                    h::Float32, hA::Float32, g::Float32, gA::Float32; transverse=false)
    
    N_total = N + 1
    results = Dict{Int, Tuple{Float32, Vector{Float32},Vector{Float32}}}()
    
    ψ_init = initialize_plus_y_state(N_total)
    
    for (i,λ) in enumerate(λ_vals)   
        tmax = 500.0f0 / λ
        dt = 0.5f0 / λ
        
        times = collect(0.0f0:dt:tmax)
        t_steps = length(times)

        H = transverse ? 
            build_hamiltonian_transverse(N, J, λ, h, hA, g, gA) : 
            build_hamiltonian(N, J, λ, h, hA, g, gA)
        
        mi_trace = Vector{Float32}(undef, t_steps)
        ψ = copy(ψ_init)
        mi_trace[1] = compute_mutual_information(ψ, N)
        
        for step in 2:t_steps
            ψ, _ = exponentiate(H, -im * dt, ψ; ishermitian=true, krylovdim=30, tol=1e-7)
            mi_trace[step] = compute_mutual_information(ψ, N)
        end
        
        results[i] = (λ, times, mi_trace)
    end
    
    return results
end;

In [ ]:
fig = Figure(size = (1000, 450), fontsize = 14)

ax1 = Axis(fig[1, 1], title = "",
    xlabel = L"\lambda t", 
    ylabel = L"I(t)",
    limits = (0, 50, nothing, nothing)
)
ax2 = Axis(fig[1, 2], title = "",
    xlabel = L"\lambda t", xscale = log10, 
    ylabel = L"\bar{I} (t)",
    limits = (5e-1, 500, nothing, nothing),
)

N = 12              
J = 1.0f0
h = 1.05f0
hA = h
g = 0.45f0
gA = g

λ_values = Float32.(10 .^ collect(-2.:0.5:2.))
colors = [(Makie.resample_cmap(:plasma, length(λ_values))...)]

results = simulate_mutual_information(N, J, λ_values, h, hA, g, gA; transverse=true)

for (c, (λ, times, mi)) in results
    cum_avg = similar(mi)
    running_sum = 0.0f0
    for i in eachindex(mi)
        running_sum += mi[i]
        cum_avg[i] = running_sum / i
    end
    
    lines!(ax1, λ .* times, mi, color = colors[c], linewidth = 2)
    lines!(ax2, λ .* times, cum_avg, color = colors[c], linewidth = 2)
end

Colorbar(fig[1, 3], limits = (10^(-2), 10^(2)), colormap = :plasma, label = L"\lambda", scale = log10)

display(fig)